# Government QA Model Evaluation

Public portfolio edition prepared for GitHub and Databricks. Credentials are read from environment variables; research data and generated artifacts are not committed to Git.


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")

from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

In [ ]:
# Databricks uses Unity Catalog Volumes; no Google Drive mount is required.

In [ ]:
!pip -q install ragas "datasets>=2.19" langchain-community langchain-openai \
                 sentence-transformers transformers tiktoken rouge-score --upgrade

import os, re, json, ast, numpy as np, pandas as pd, torch, math, time
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from functools import lru_cache
import multiprocessing as mp

In [ ]:
# -------- 路径（可改）--------
P_IN       = Path("/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_all_models_unified.1.1.parquet")
OUT_RETR   = Path("/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_scored.retrieval.1.1.parquet")
OUT_ORC    = Path("/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_scored.oracle.1.1.parquet")
OUT_FULL   = Path("/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_scored.full.1.1.parquet")
TMP_DIR    = Path("/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/_tmp_govern_rows")
TMP_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# -------- 优化配置 --------
MODEL_POOL_AR = ["gpt-4o-mini", "gpt-4.1-nano", "gpt-4.1-mini", "gpt-3.5-turbo"]
MODEL_POOL_FA = ["gpt-4o-mini", "gpt-4.1-nano", "gpt-4.1-mini", "gpt-3.5-turbo"]
BATCH_SAVE_EVERY = 50  # 更频繁保存，减少内存占用
MAX_WORKERS = min(4, mp.cpu_count())  # 并行处理
EMBEDDING_BATCH_SIZE = 64  # 优化embedding批次大小

In [ ]:
# -------- 读入 + 保障类型（优化版）--------
print("📖 Loading data...")
DF = pd.read_parquet(P_IN)

# 使用更高效的类型转换
string_cols = ["question","answer","gold","paper_id","question_id","question_norm","model"]
for c in string_cols:
    if c in DF.columns:
        DF[c] = DF[c].fillna("").astype("string")  # 使用pandas string dtype，更省内存


In [ ]:
@lru_cache(maxsize=1000)  # 缓存常见的转换结果
def _ensure_list_str_cached(x_str):
    """缓存版本的列表字符串转换"""
    if not x_str or x_str == "nan":
        return []

    # 尝试JSON解析
    if x_str.startswith("[") and x_str.endswith("]"):
        try:
            obj = json.loads(x_str)
            if isinstance(obj, list):
                return [str(t) for t in obj if str(t).strip()]
        except:
            try:
                obj = ast.literal_eval(x_str)
                if isinstance(obj, (list, tuple)):
                    return [str(t) for t in obj if str(t).strip()]
            except:
                pass
    return [x_str] if x_str.strip() else []

def _ensure_list_str(x):
    """优化的列表字符串转换"""
    if isinstance(x, (list, tuple)):
        return [str(t) for t in x if str(t).strip()]
    if hasattr(x, 'tolist'):  # numpy array
        return [str(t) for t in x.tolist() if str(t).strip()]
    if x is None:
        return []

    return _ensure_list_str_cached(str(x))

In [ ]:
# 批量处理列转换
list_cols = ["oracle_ctx","retrieved_ctx","citations","_ref_topk","_ctx_topk"]
for col in list_cols:
    if col in DF.columns:
        print(f"Processing {col}...")
        DF[col] = DF[col].apply(_ensure_list_str)
    elif col == "citations":
        DF[col] = [[] for _ in range(len(DF))]


In [ ]:
# -------- 优化的token处理 --------
import tiktoken
from rouge_score import rouge_scorer

ENC = tiktoken.get_encoding("cl100k_base")

@lru_cache(maxsize=5000)  # 缓存token长度计算
def _tok_len_cached(s):
    return len(ENC.encode(s or ""))

@lru_cache(maxsize=2000)  # 缓存截断结果
def _truncate_cached(s, max_toks):
    if not s:
        return ""
    ids = ENC.encode(s)
    return s if len(ids) <= max_toks else ENC.decode(ids[:max_toks])

_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

def _select_topk_optimized(evs, q, a, k=3, budget=500, per_ctx=180):
    """优化的top-k选择"""
    if not evs:
        return []

    # 预筛选非空项
    valid_evs = [e for e in evs if isinstance(e, str) and len(e.strip()) > 10]
    if not valid_evs:
        return []

    # 向量化ROUGE计算（如果项目较多）
    if len(valid_evs) > 20:
        # 对于大量候选，先用简单启发式筛选
        scored = []
        q_words = set(q.lower().split())
        a_words = set(a.lower().split())

        for ev in valid_evs:
            ev_words = set(ev.lower().split())
            overlap_q = len(q_words & ev_words) / max(len(q_words), 1)
            overlap_a = len(a_words & ev_words) / max(len(a_words), 1)
            simple_score = 0.6 * overlap_q + 0.4 * overlap_a
            scored.append((simple_score, ev))

        # 取top候选再用ROUGE精确计算
        scored.sort(key=lambda x: x[0], reverse=True)
        top_candidates = [ev for _, ev in scored[:k*2]]
    else:
        top_candidates = valid_evs

    # ROUGE精确评分
    rouge_scored = []
    for ev in top_candidates:
        try:
            rouge_q = _scorer.score(q, ev)["rougeL"].fmeasure
            rouge_a = _scorer.score(a, ev)["rougeL"].fmeasure
            score = 0.6 * rouge_q + 0.4 * rouge_a
            rouge_scored.append((score, ev))
        except:
            rouge_scored.append((0.0, ev))

    rouge_scored.sort(key=lambda x: x[0], reverse=True)

    # 预算管理
    out, tot = [], 0
    for _, ev in rouge_scored[:k*3]:
        ev2 = _truncate_cached(ev, per_ctx)
        t = _tok_len_cached(ev2)
        if tot + t <= budget:
            out.append(ev2)
            tot += t
        if len(out) >= k:
            break

    if not out and valid_evs:
        out = [_truncate_cached(valid_evs[0], min(300, budget))]

    return out[:k]

In [ ]:
# 并行构建top-k
if "_ref_topk" not in DF.columns or DF["_ref_topk"].apply(len).eq(0).any():
    print("🔄 Building reference top-k...")
    def build_ref_topk(args):
        i, row = args
        return i, _select_topk_optimized(row["oracle_ctx"], row["question"], row["answer"])

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        results = list(tqdm(executor.map(build_ref_topk, DF.iterrows()),
                           total=len(DF), desc="Building _ref_topk"))

    ref_topk_dict = {i: result for i, result in results}
    DF["_ref_topk"] = [ref_topk_dict.get(i, []) for i in DF.index]

if "_ctx_topk" not in DF.columns or DF["_ctx_topk"].apply(len).eq(0).any():
    print("🔄 Building context top-k...")
    def build_ctx_topk(args):
        i, row = args
        return i, _select_topk_optimized(row["retrieved_ctx"], row["question"], row["answer"])

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        results = list(tqdm(executor.map(build_ctx_topk, DF.iterrows()),
                           total=len(DF), desc="Building _ctx_topk"))

    ctx_topk_dict = {i: result for i, result in results}
    DF["_ctx_topk"] = [ctx_topk_dict.get(i, []) for i in DF.index]


In [ ]:
# -------- 优化的RAGAS评估 --------
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_relevancy as M_ANS_REL
from ragas.metrics import faithfulness as M_FAITH

try:
    from ragas.metrics import context_precision as M_CP, context_recall as M_CR
except ImportError:
    M_CP = M_CR = None

from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer, util as st_util


In [ ]:
# 全局模型初始化（避免重复加载）
print("🤖 Initializing models...")
emb = HuggingFaceEmbeddings(model_name="intfloat/e5-base-v2")
device = "cuda" if torch.cuda.is_available() else "cpu"
e5 = SentenceTransformer("intfloat/e5-base-v2", device=device)


In [ ]:
# LLM池化机制优化
class LLMPool:
    def __init__(self):
        self._json_llms = {}
        self._plain_llms = {}

    def get_json_llm(self, model):
        if model not in self._json_llms:
            try:
                self._json_llms[model] = ChatOpenAI(
                    model=model, temperature=0,
                    model_kwargs={"response_format": {"type": "json_object"}},
                    request_timeout=60
                )
            except:
                self._json_llms[model] = ChatOpenAI(model=model, temperature=0, request_timeout=60)
        return self._json_llms[model]

    def get_plain_llm(self, model):
        if model not in self._plain_llms:
            self._plain_llms[model] = ChatOpenAI(model=model, temperature=0, request_timeout=60)
        return self._plain_llms[model]

llm_pool = LLMPool()

def _join_ref(lst):
    return "\n\n".join([x for x in (lst or []) if isinstance(x, str) and x.strip()])

def _eval_batch_oracle(batch_rows):
    """批量评估Oracle指标"""
    results = []

    for row in batch_rows:
        ar = np.nan
        # Answer Relevancy with model pooling
        for m in MODEL_POOL_AR:
            try:
                ds = Dataset.from_dict({
                    "user_input": [row["question"]],
                    "response": [row["answer"]],
                    "retrieved_contexts": [row["_ref_topk"]]
                })
                out = evaluate(ds, metrics=[M_ANS_REL],
                             llm=llm_pool.get_json_llm(m), embeddings=emb).to_pandas()
                v = out.loc[0, "answer_relevancy"]
                if not (isinstance(v, float) and np.isnan(v)):
                    ar = float(v)
                    break
            except Exception as e:
                continue

        # Faithfulness
        fa = np.nan
        for m in MODEL_POOL_FA:
            try:
                ds = Dataset.from_dict({
                    "user_input": [row["question"]],
                    "response": [row["answer"]],
                    "retrieved_contexts": [row["_ref_topk"]]
                })
                out = evaluate(ds, metrics=[M_FAITH],
                             llm=llm_pool.get_plain_llm(m), embeddings=emb).to_pandas()
                v = out.loc[0, "faithfulness"]
                if not (isinstance(v, float) and np.isnan(v)):
                    fa = float(v)
                    break
            except Exception:
                continue

        # E5 fallback for AR
        if isinstance(ar, float) and np.isnan(ar):
            try:
                qa = e5.encode([f"query: {row['question']}", f"passage: {row['answer']}"],
                              normalize_embeddings=True, show_progress_bar=False)
                sim = float(st_util.cos_sim(qa[0], qa[1])[0][0].item())
                ar = (sim + 1.0) / 2.0
            except:
                ar = np.nan

        results.append({
            "oracle::answer_relevancy": ar,
            "oracle::faithfulness": fa
        })

    return results

def _eval_batch_retrieval(batch_rows):
    """批量评估Retrieval指标"""
    results = []

    for row in batch_rows:
        if not (isinstance(row["_ctx_topk"], (list, tuple)) and len(row["_ctx_topk"]) > 0):
            results.append({
                "retrieval::answer_relevancy": np.nan,
                "retrieval::faithfulness": np.nan,
                "retrieval::context_precision": np.nan,
                "retrieval::context_recall": np.nan
            })
            continue

        # Answer Relevancy
        ar = np.nan
        for m in MODEL_POOL_AR:
            try:
                ds = Dataset.from_dict({
                    "user_input": [row["question"]],
                    "response": [row["answer"]],
                    "retrieved_contexts": [row["_ctx_topk"]],
                    "reference": [_join_ref(row["_ref_topk"])]
                })
                out = evaluate(ds, metrics=[M_ANS_REL],
                             llm=llm_pool.get_json_llm(m), embeddings=emb).to_pandas()
                v = out.loc[0, "answer_relevancy"]
                if not (isinstance(v, float) and np.isnan(v)):
                    ar = float(v)
                    break
            except Exception:
                continue

        # Faithfulness
        fa = np.nan
        for m in MODEL_POOL_FA:
            try:
                ds = Dataset.from_dict({
                    "user_input": [row["question"]],
                    "response": [row["answer"]],
                    "retrieved_contexts": [row["_ctx_topk"]],
                    "reference": [_join_ref(row["_ref_topk"])]
                })
                out = evaluate(ds, metrics=[M_FAITH],
                             llm=llm_pool.get_plain_llm(m), embeddings=emb).to_pandas()
                v = out.loc[0, "faithfulness"]
                if not (isinstance(v, float) and np.isnan(v)):
                    fa = float(v)
                    break
            except Exception:
                continue

        # Context Precision/Recall
        cp = cr = np.nan
        if M_CP is not None or M_CR is not None:
            try:
                mets = [m for m in [M_CP, M_CR] if m is not None]
                ds2 = Dataset.from_dict({
                    "user_input": [row["question"]],
                    "response": [row["answer"]],
                    "retrieved_contexts": [row["_ctx_topk"]],
                    "reference": [_join_ref(row["_ref_topk"])]
                })
                out2 = evaluate(ds2, metrics=mets,
                              llm=llm_pool.get_plain_llm(MODEL_POOL_FA[0]),
                              embeddings=emb).to_pandas()
                if "context_precision" in out2.columns:
                    cp = float(out2.loc[0, "context_precision"])
                if "context_recall" in out2.columns:
                    cr = float(out2.loc[0, "context_recall"])
            except Exception:
                pass

        # E5 fallback for AR
        if isinstance(ar, float) and np.isnan(ar):
            try:
                qa = e5.encode([f"query: {row['question']}", f"passage: {row['answer']}"],
                              normalize_embeddings=True, show_progress_bar=False)
                sim = float(st_util.cos_sim(qa[0], qa[1])[0][0].item())
                ar = (sim + 1.0) / 2.0
            except:
                ar = np.nan

        results.append({
            "retrieval::answer_relevancy": ar,
            "retrieval::faithfulness": fa,
            "retrieval::context_precision": cp,
            "retrieval::context_recall": cr
        })

    return results

In [ ]:
# -------- 断点续跑优化 --------
def _seed_from(path, cols):
    if path.exists():
        try:
            old = pd.read_parquet(path)
            inter = [c for c in cols if c in old.columns]
            print(f"⏭️  载入已有结果: {path.name} -> {len(old)} 行, 列 {inter}")
            return old[inter]
        except Exception as e:
            print(f"⚠️ 无法读取旧结果：{e}")
    return None

# 初始化结果列
result_cols = [
    "oracle::answer_relevancy", "oracle::faithfulness",
    "retrieval::answer_relevancy", "retrieval::faithfulness",
    "retrieval::context_precision", "retrieval::context_recall"
]

for c in result_cols:
    if c not in DF.columns:
        DF[c] = np.nan

In [ ]:
# -------- 执行：Retrieval（批量优化版）--------
mask_r = DF["_ctx_topk"].apply(lambda x: isinstance(x, (list, tuple)) and len(x) > 0)
IDX_R = DF.index[mask_r].tolist()
print(f"▶️ Retrieval rows: {len(IDX_R)}")

# 恢复已有结果
seed = _seed_from(OUT_RETR, ["retrieval::answer_relevancy", "retrieval::faithfulness",
                            "retrieval::context_precision", "retrieval::context_recall"])
if seed is not None and len(seed) == len(DF):
    for c in seed.columns:
        DF[c] = seed[c].values

pending_r = [i for i in IDX_R if any(pd.isna(DF.loc[i, c]) for c in
                                    ["retrieval::answer_relevancy", "retrieval::faithfulness"])]

print(f"🔄 Processing {len(pending_r)} pending retrieval rows...")


In [ ]:
def load_existing_retrieval_results():
    """加载已有的retrieval评估结果"""
    EXISTING_RETR = Path("/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_scored.retrieval.1.1.parquet")

    if EXISTING_RETR.exists():
        print(f"🎯 Loading existing retrieval results from {EXISTING_RETR.name}...")
        try:
            existing_results = pd.read_parquet(EXISTING_RETR)
            print(f"📊 Existing results: {len(existing_results)} rows")

            # 检查列匹配
            retrieval_cols = ["retrieval::answer_relevancy", "retrieval::faithfulness",
                             "retrieval::context_precision", "retrieval::context_recall"]
            available_cols = [c for c in retrieval_cols if c in existing_results.columns]
            print(f"📋 Available columns: {available_cols}")

            # 如果数据框长度匹配，直接使用索引对齐
            if len(existing_results) == len(DF):
                print("✅ Length matches - using direct index alignment")
                for c in available_cols:
                    DF[c] = existing_results[c].values

                # 统计已完成的工作
                completed_count = DF["retrieval::answer_relevancy"].notna().sum()
                print(f"🎉 Loaded {completed_count} existing retrieval evaluations ({completed_count/len(DF)*100:.1f}%)")
                return True

            else:
                # 尝试基于关键列进行匹配
                key_cols = ['question', 'answer', 'model']  # 用于匹配的关键列
                available_keys = [c for c in key_cols if c in existing_results.columns and c in DF.columns]

                if available_keys:
                    print(f"🔗 Merging based on keys: {available_keys}")
                    # 创建合并键
                    existing_results['_merge_key'] = existing_results[available_keys].astype(str).agg('_'.join, axis=1)
                    DF['_merge_key'] = DF[available_keys].astype(str).agg('_'.join, axis=1)

                    # 合并数据
                    merge_df = DF[['_merge_key']].merge(
                        existing_results[['_merge_key'] + available_cols],
                        on='_merge_key', how='left'
                    )

                    # 更新DF
                    for c in available_cols:
                        DF[c] = merge_df[c].values

                    # 清理临时列
                    DF.drop('_merge_key', axis=1, inplace=True)

                    matched_count = merge_df[available_cols[0]].notna().sum()
                    print(f"✅ Successfully matched {matched_count}/{len(DF)} rows ({matched_count/len(DF)*100:.1f}%)")
                    return True
                else:
                    print("⚠️ No matching keys found - will recompute all")
                    return False

        except Exception as e:
            print(f"❌ Error loading existing results: {e}")
            return False
    else:
        print("ℹ️ No existing 1.1 results found")
        return False


In [ ]:
# 初始化结果列
retrieval_cols = ["retrieval::answer_relevancy", "retrieval::faithfulness",
                  "retrieval::context_precision", "retrieval::context_recall"]
for c in retrieval_cols:
    if c not in DF.columns:
        DF[c] = np.nan

In [ ]:
# 加载已有结果
loaded_existing = load_existing_retrieval_results()

In [ ]:
mask_r = DF["_ctx_topk"].apply(lambda x: isinstance(x, (list, tuple)) and len(x) > 0)
IDX_R = DF.index[mask_r].tolist()
pending_r = [i for i in IDX_R if any(pd.isna(DF.loc[i, c]) for c in
                                    ["retrieval::answer_relevancy", "retrieval::faithfulness"])]

completed_r = len(IDX_R) - len(pending_r)
print(f"\n📊 Retrieval Status:")
print(f"   Total retrieval rows: {len(IDX_R)}")
print(f"   Completed: {completed_r} ({completed_r/len(IDX_R)*100:.1f}%)")
print(f"   Remaining: {len(pending_r)} ({len(pending_r)/len(IDX_R)*100:.1f}%)")
print(f"   Time saved: ~{completed_r * 30} seconds (估算)")  # 假设每行30秒

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_relevancy as M_ANS_REL
from ragas.metrics import faithfulness as M_FAITH

try:
    from ragas.metrics import context_precision as M_CP, context_recall as M_CR
except ImportError:
    M_CP = M_CR = None
    print("⚠️ Context precision/recall not available in this RAGAS version")

from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer, util as st_util

In [ ]:
# 初始化模型（如果还没有）
if 'emb' not in globals():
    print("🤖 Initializing models...")
    emb = HuggingFaceEmbeddings(model_name="intfloat/e5-base-v2")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    e5 = SentenceTransformer("intfloat/e5-base-v2", device=device)


In [ ]:
# LLM池化机制
class LLMPool:
    def __init__(self):
        self._json_llms = {}
        self._plain_llms = {}

    def get_json_llm(self, model):
        if model not in self._json_llms:
            try:
                self._json_llms[model] = ChatOpenAI(
                    model=model, temperature=0,
                    model_kwargs={"response_format": {"type": "json_object"}},
                    request_timeout=60
                )
            except:
                self._json_llms[model] = ChatOpenAI(model=model, temperature=0, request_timeout=60)
        return self._json_llms[model]

    def get_plain_llm(self, model):
        if model not in self._plain_llms:
            self._plain_llms[model] = ChatOpenAI(model=model, temperature=0, request_timeout=60)
        return self._plain_llms[model]

llm_pool = LLMPool()

In [ ]:
def _join_ref(lst):
    return "\n\n".join([x for x in (lst or []) if isinstance(x, str) and x.strip()])

def _eval_row_retrieval_optimized(row):
    """优化的单行retrieval评估"""
    if not (isinstance(row["_ctx_topk"], (list, tuple)) and len(row["_ctx_topk"]) > 0):
        return {
            "retrieval::answer_relevancy": np.nan,
            "retrieval::faithfulness": np.nan,
            "retrieval::context_precision": np.nan,
            "retrieval::context_recall": np.nan
        }

    # Answer Relevancy (with model fallback)
    ar = np.nan
    for m in MODEL_POOL_AR:
        try:
            ds = Dataset.from_dict({
                "user_input": [row["question"]],
                "response": [row["answer"]],
                "retrieved_contexts": [row["_ctx_topk"]],
                "reference": [_join_ref(row["_ref_topk"])]
            })
            out = evaluate(ds, metrics=[M_ANS_REL],
                         llm=llm_pool.get_json_llm(m), embeddings=emb).to_pandas()
            v = out.loc[0, "answer_relevancy"]
            if not (isinstance(v, float) and np.isnan(v)):
                ar = float(v)
                break
        except Exception as e:
            print(f"⚠️ AR failed with {m}: {str(e)[:100]}...")
            continue

    # Faithfulness (with model fallback)
    fa = np.nan
    for m in MODEL_POOL_FA:
        try:
            ds = Dataset.from_dict({
                "user_input": [row["question"]],
                "response": [row["answer"]],
                "retrieved_contexts": [row["_ctx_topk"]],
                "reference": [_join_ref(row["_ref_topk"])]
            })
            out = evaluate(ds, metrics=[M_FAITH],
                         llm=llm_pool.get_plain_llm(m), embeddings=emb).to_pandas()
            v = out.loc[0, "faithfulness"]
            if not (isinstance(v, float) and np.isnan(v)):
                fa = float(v)
                break
        except Exception as e:
            print(f"⚠️ Faithfulness failed with {m}: {str(e)[:100]}...")
            continue

    # Context Precision/Recall (if available)
    cp = cr = np.nan
    if M_CP is not None or M_CR is not None:
        try:
            mets = [m for m in [M_CP, M_CR] if m is not None]
            ds2 = Dataset.from_dict({
                "user_input": [row["question"]],
                "response": [row["answer"]],
                "retrieved_contexts": [row["_ctx_topk"]],
                "reference": [_join_ref(row["_ref_topk"])]
            })
            out2 = evaluate(ds2, metrics=mets,
                          llm=llm_pool.get_plain_llm(MODEL_POOL_FA[0]),
                          embeddings=emb).to_pandas()
            if "context_precision" in out2.columns:
                cp = float(out2.loc[0, "context_precision"])
            if "context_recall" in out2.columns:
                cr = float(out2.loc[0, "context_recall"])
        except Exception as e:
            print(f"⚠️ Context metrics failed: {str(e)[:100]}...")

    # E5 fallback for Answer Relevancy
    if isinstance(ar, float) and np.isnan(ar):
        try:
            qa = e5.encode([f"query: {row['question']}", f"passage: {row['answer']}"],
                          normalize_embeddings=True, show_progress_bar=False)
            sim = float(st_util.cos_sim(qa[0], qa[1])[0][0].item())
            ar = (sim + 1.0) / 2.0
            print(f"🔄 Used E5 fallback for AR: {ar:.3f}")
        except Exception as e:
            print(f"❌ E5 fallback failed: {e}")
            ar = np.nan

    return {
        "retrieval::answer_relevancy": ar,
        "retrieval::faithfulness": fa,
        "retrieval::context_precision": cp,
        "retrieval::context_recall": cr
    }

In [ ]:
# 找出需要处理的行
mask_r = DF["_ctx_topk"].apply(lambda x: isinstance(x, (list, tuple)) and len(x) > 0)
IDX_R = DF.index[mask_r].tolist()
pending_r = [i for i in IDX_R if any(pd.isna(DF.loc[i, c]) for c in
                                    ["retrieval::answer_relevancy", "retrieval::faithfulness"])]

print(f"🔄 Processing {len(pending_r)} remaining retrieval rows...")
print(f"📊 Progress: {len(IDX_R) - len(pending_r)}/{len(IDX_R)} completed ({(len(IDX_R) - len(pending_r))/len(IDX_R)*100:.1f}%)")


In [ ]:
# 处理剩余行（带进度保存）
start_time = time.time()
for k, i in enumerate(tqdm(pending_r, desc="RAGAS retrieval evaluation")):
    try:
        row = DF.loc[i]
        result = _eval_row_retrieval_optimized(row)

        # 更新结果
        for col, val in result.items():
            DF.loc[i, col] = val

        # 定期保存进度
        if (k + 1) % BATCH_SAVE_EVERY == 0:
            DF.to_parquet(OUT_RETR, index=False)
            elapsed = time.time() - start_time
            avg_time = elapsed / (k + 1)
            remaining_time = avg_time * (len(pending_r) - k - 1)
            print(f"💾 Saved progress: {k+1}/{len(pending_r)} rows ({(k+1)/len(pending_r)*100:.1f}%)")
            print(f"⏱️  ETA: {remaining_time/60:.1f} minutes")

        # 显示中间结果
        if (k + 1) % 10 == 0:
            ar_val = result["retrieval::answer_relevancy"]
            fa_val = result["retrieval::faithfulness"]
            print(f"📈 Row {i}: AR={ar_val:.3f if not pd.isna(ar_val) else 'N/A'}, "
                  f"FA={fa_val:.3f if not pd.isna(fa_val) else 'N/A'}")

    except Exception as e:
        print(f"❌ Error processing row {i}: {e}")
        # 设置为NaN以避免重复处理
        for col in ["retrieval::answer_relevancy", "retrieval::faithfulness",
                   "retrieval::context_precision", "retrieval::context_recall"]:
            DF.loc[i, col] = np.nan

# 最终保存
DF.to_parquet(OUT_RETR, index=False)


In [ ]:
# 统计结果
completed_after = DF["retrieval::answer_relevancy"].notna().sum()
total_time = time.time() - start_time

print(f"\n🎉 Retrieval evaluation completed!")
print(f"📊 Final stats:")
print(f"   Completed rows: {completed_after}/{len(IDX_R)} ({completed_after/len(IDX_R)*100:.1f}%)")
print(f"   Newly processed: {len(pending_r)}")
print(f"   Total time: {total_time/60:.1f} minutes")
print(f"   Average time per row: {total_time/len(pending_r):.1f} seconds")
print(f"💾 Results saved to: {OUT_RETR}")

# 显示一些样本结果
print(f"\n📈 Sample results:")
sample_cols = ["retrieval::answer_relevancy", "retrieval::faithfulness"]
existing_cols = [c for c in sample_cols if c in DF.columns]
if existing_cols:
    print(DF[existing_cols].describe().round(3))

In [ ]:
# -------- 执行：Oracle（批量优化版）--------
IDX_O = DF.index.tolist()
print(f"▶️ Oracle rows: {len(IDX_O)}")

seed = _seed_from(OUT_ORC, ["oracle::answer_relevancy", "oracle::faithfulness"])
if seed is not None and len(seed) == len(DF):
    for c in seed.columns:
        DF[c] = seed[c].values

pending_o = [i for i in IDX_O if any(pd.isna(DF.loc[i, c]) for c in
                                    ["oracle::answer_relevancy", "oracle::faithfulness"])]

print(f"🔄 Processing {len(pending_o)} pending oracle rows...")

In [ ]:
EVAL_BATCH_SIZE = 12

In [ ]:
# 批量处理Oracle
for start_idx in tqdm(range(0, len(pending_o), EVAL_BATCH_SIZE), desc="RAGAS oracle batches"):
    batch_indices = pending_o[start_idx:start_idx + EVAL_BATCH_SIZE]
    batch_rows = [DF.loc[i] for i in batch_indices]

    try:
        batch_results = _eval_batch_oracle(batch_rows)

        for idx, result in zip(batch_indices, batch_results):
            for col, val in result.items():
                DF.loc[idx, col] = val

        if (start_idx + EVAL_BATCH_SIZE) % (BATCH_SAVE_EVERY * EVAL_BATCH_SIZE) == 0:
            DF.to_parquet(OUT_ORC, index=False)
            print(f"💾 Saved progress at batch {start_idx // EVAL_BATCH_SIZE + 1}")

    except Exception as e:
        print(f"❌ Error in batch {start_idx // EVAL_BATCH_SIZE + 1}: {e}")
        for idx in batch_indices:
            try:
                row = DF.loc[idx]
                result = _eval_batch_oracle([row])[0]
                for col, val in result.items():
                    DF.loc[idx, col] = val
            except Exception as e2:
                print(f"❌ Error in row {idx}: {e2}")

DF.to_parquet(OUT_ORC, index=False)
print("💾 Saved oracle metrics:", OUT_ORC)


In [ ]:
# ========= 加速版 Oracle 评审主循环（异步并发 + 原子落盘 + 去重） =========
import asyncio, json, time, random, hashlib, os, tempfile, shutil, atexit
from typing import Dict, Any, List, Tuple
import pandas as pd

# ---------------- 参数区（可按你的环境调整） ----------------
ORACLE_MODEL          = "gpt-4o-mini", "gpt-4.1-nano", "gpt-4.1-mini", "gpt-3.5-turbo"     # 你当前用于 oracle 评审的快速模型
CONCURRENCY           = 12                # 并发度，Colab/T4 可 8-12；A100 可 16-24
MAX_RETRIES           = 5                 # 单条最大重试次数
RETRY_BASE_DELAY      = 1.2               # 初始重试等待（秒），指数回退
SAVE_EVERY_N_ROWS     = 100               # 每处理 N 条原子写盘一次
SAVE_EVERY_SECONDS    = 90                # 或每隔 T 秒强制写盘一次（二选一满足其一就落盘）
WRITE_JSONL_JOURNAL   = True              # 是否追加写 JSONL 日志（崩溃也能保留最新进度）
OUT_ORC_JSONL         = str(os.path.splitext(OUT_ORC)[0]) + ".journal.jsonl"  # 日志路径


In [ ]:
# ---------------- 工具函数 ----------------
def _atomic_write_parquet(df: pd.DataFrame, path: str):
    """原子落盘：先写临时文件，再替换目标文件，防止半写入损坏。"""
    tmpdir = tempfile.mkdtemp(prefix="orc_tmp_")
    tmppath = os.path.join(tmpdir, "part.parquet")
    try:
        df.to_parquet(tmppath, index=False)
        shutil.move(tmppath, path)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)

def _append_jsonl(path: str, record: Dict[str, Any]):
    """安全 JSONL 追加。"""
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def _now_ms():
    return int(time.time() * 1000)

def _row_cache_key(row: pd.Series) -> str:
    """
    针对同一问题/同一答案/同一证据内容生成稳定 key，避免重复评审。
    你可以按自己列名微调组合字段：
    - qid / question
    - 评审目标答案列（如 answer_M3 或你当前在做的模型答案列）
    - oracle 或 retrieved context 的唯一标识（如 sha256 / 文本本体）
    """
    parts = [
        str(row.get("qid", "")),
        str(row.get("question", "")),
        # 猜测你把不同模型答案存在不同列，这里优先 M3；可换成你当前要评审的答案列名
        str(row.get("answer_M3", row.get("answer", ""))),
        # 你数据里已有 topk_sha256；oracle 评审建议用 oracle 证据的 hash
        str(row.get("oracle_sha256", row.get("topk_sha256", ""))),
    ]
    s = "||".join(parts)
    return hashlib.sha1(s.encode("utf-8")).hexdigest()

def _build_oracle_messages(row: pd.Series) -> List[Dict[str, str]]:
    """
    使用“单轮 JSON 输出”的提示词。若你已有自定义 prompt，直接替换本函数的内容即可。
    要求返回严格 JSON（含两个 0~1 的分数）。
    """
    question = str(row.get("question", "")).strip()
    answer   = str(row.get("answer_M3", row.get("answer", ""))).strip()
    # 这里使用 oracle 证据；若你当前做的是 retrieved 评审，请改成相应字段
    evidence = str(row.get("oracle_context", row.get("retrieved_context", ""))).strip()

    sys = (
        "You are a careful research assistant. "
        "Judge the model's answer using ONLY the provided evidence. "
        "Return a JSON object with numeric fields 'answer_relevancy' and 'faithfulness', each in [0,1]."
    )
    user = (
        f"Question:\n{question}\n\n"
        f"Model Answer:\n{answer}\n\n"
        f"Evidence (ground truth context):\n{evidence}\n\n"
        "Definitions:\n"
        "- answer_relevancy: Does the answer directly address the question? 1=yes, 0=no.\n"
        "- faithfulness: Is every factual claim in the answer supported by the evidence? 1=fully supported, 0=unsupported.\n\n"
        "STRICTLY RETURN JSON like: {\"answer_relevancy\": 0.0-1.0, \"faithfulness\": 0.0-1.0}."
    )
    return [{"role":"system","content":sys}, {"role":"user","content":user}]

def _parse_oracle_json(text: str) -> Tuple[float, float]:
    """鲁棒解析：无论是否严格 JSON 模式，都尽力解析出两个分数。"""
    try:
        obj = json.loads(text)
        ar = float(obj.get("answer_relevancy"))
        ft = float(obj.get("faithfulness"))
        # 裁剪到 [0,1]
        ar = max(0.0, min(1.0, ar))
        ft = max(0.0, min(1.0, ft))
        return ar, ft
    except Exception:
        # 简单兜底：用正则或关键词提取，这里给最小兜底 0/0
        return 0.0, 0.0

async def _call_llm_json(llm, messages: List[Dict[str,str]]) -> Tuple[float,float,str]:
    """
    带重试的 LLM 调用。优先 JSON 模式，不支持则退回纯文本后自行解析。
    返回: (answer_relevancy, faithfulness, raw_text)
    """
    delay = RETRY_BASE_DELAY
    for attempt in range(1, MAX_RETRIES+1):
        try:
            # LangChain ChatOpenAI 统一 ainvoke
            resp = await llm.ainvoke(messages)
            # 兼容不同返回结构
            raw = getattr(resp, "content", None) or str(resp)
            ar, ft = _parse_oracle_json(raw)
            return ar, ft, raw
        except Exception as e:
            if attempt >= MAX_RETRIES:
                # 最终失败，返回 0/0 并携带错误信息
                return 0.0, 0.0, f"ERROR:{type(e).__name__}:{e}"
            await asyncio.sleep(delay + random.random())
            delay = min(delay * 2, 30.0)

In [ ]:
async def _evaluate_rows_async(indices: List[int], df: pd.DataFrame) -> None:
    """
    核心：并发评审 + 去重缓存 + 增量写盘
    直接就地更新 df['oracle::answer_relevancy'] / df['oracle::faithfulness']
    """
    llm = llm_pool.get_json_llm(ORACLE_MODEL)  # 先尝试 JSON 模式；若失败池里会降级

    cache: Dict[str, Tuple[float,float]] = {}
    # 若之前已有种子结果（你上面 seed 代码已把旧值灌回 DF 了），也用来热启动缓存
    for i in indices:
        row = df.loc[i]
        if not pd.isna(row.get("oracle::answer_relevancy")) and not pd.isna(row.get("oracle::faithfulness")):
            cache[_row_cache_key(row)] = (float(row["oracle::answer_relevancy"]), float(row["oracle::faithfulness"]))

    n_done = 0
    last_save_ms = _now_ms()
    lock = asyncio.Lock()            # 保护写 DF/落盘
    sem  = asyncio.Semaphore(CONCURRENCY)

    async def worker(idx: int):
        nonlocal n_done, last_save_ms
        row = df.loc[idx]
        key = _row_cache_key(row)

        if key in cache:
            ar, ft = cache[key]
            async with lock:
                df.loc[idx, "oracle::answer_relevancy"] = ar
                df.loc[idx, "oracle::faithfulness"] = ft
                n_done += 1
            return

        messages = _build_oracle_messages(row)

        async with sem:
            ar, ft, raw = await _call_llm_json(llm, messages)

        cache[key] = (ar, ft)
        async with lock:
            df.loc[idx, "oracle::answer_relevancy"] = ar
            df.loc[idx, "oracle::faithfulness"] = ft
            n_done += 1

            # 写 JSONL 日志（极小 IO，崩溃也不丢失）
            if WRITE_JSONL_JOURNAL:
                _append_jsonl(OUT_ORC_JSONL, {
                    "idx": idx,
                    "answer_relevancy": ar,
                    "faithfulness": ft,
                    "ts": int(time.time()),
                })

            # 触发增量原子落盘
            need_by_count = (n_done % SAVE_EVERY_N_ROWS == 0)
            need_by_time  = (_now_ms() - last_save_ms) >= (SAVE_EVERY_SECONDS * 1000)
            if need_by_count or need_by_time:
                _atomic_write_parquet(df, OUT_ORC)
                last_save_ms = _now_ms()
                print(f"💾 Saved progress ({n_done} rows) -> {OUT_ORC}")

    # 捕获退出时的兜底保存
    def _atexit_save():
        try:
            _atomic_write_parquet(df, OUT_ORC)
            print("💾 Saved at exit ->", OUT_ORC)
        except Exception as e:
            print("⚠️ Save-at-exit failed:", e)
    atexit.register(_atexit_save)

    tasks = [asyncio.create_task(worker(i)) for i in indices]
    # 边完成边进度输出
    done_cnt = 0
    for coro in asyncio.as_completed(tasks):
        await coro
        done_cnt += 1
        if done_cnt % 20 == 0:
            print(f"✅ Progress: {done_cnt}/{len(indices)}")

    # 结束再保险保存
    _atomic_write_parquet(df, OUT_ORC)
    print("✅ All oracle metrics saved ->", OUT_ORC)

In [ ]:
# —— 放在 LLMPool 定义前 ——
def _normalize_model_name(m):
    """把传入的 model 统一成字符串（容错 tuple/list/对象）。"""
    # tuple/list -> 取第一个元素当作模型名
    if isinstance(m, (tuple, list)):
        m = m[0]
    # LangChain ChatOpenAI 实例/包装类 -> 提取其模型名（尽量不依赖内部属性）
    try:
        # 常规就是 str，这里提前返回
        if isinstance(m, str):
            return m
        # 兜底：对象上常见的属性名
        for attr in ("model_name", "model", "name"):
            if hasattr(m, attr):
                val = getattr(m, attr)
                if isinstance(val, str):
                    return val
    except Exception:
        pass
    return str(m)

class LLMPool:
    def __init__(self):
        self._json_llms = {}
        self._plain_llms = {}

    def get_json_llm(self, model):
        name = _normalize_model_name(model)
        if name not in self._json_llms:
            # o1/o3 等推理模型不要强行传 temperature=0（部分版本会报错/被改写）
            base_kwargs = dict(model=name, request_timeout=60)
            if not name.startswith(("o1", "o3")):
                base_kwargs["temperature"] = 0

            # 优先开 JSON 模式；不支持则自动回退
            try:
                # 新版 langchain-openai 支持顶层 response_format
                llm = ChatOpenAI(**base_kwargs, response_format={"type": "json_object"})
            except Exception:
                # 老版本不认识 response_format -> 去掉重试
                llm = ChatOpenAI(**base_kwargs)
            self._json_llms[name] = llm
        return self._json_llms[name]

    def get_plain_llm(self, model):
        name = _normalize_model_name(model)
        if name not in self._plain_llms:
            base_kwargs = dict(model=name, request_timeout=60)
            if not name.startswith(("o1", "o3")):
                base_kwargs["temperature"] = 0
            self._plain_llms[name] = ChatOpenAI(**base_kwargs)
        return self._plain_llms[name]

llm_pool = LLMPool()

In [ ]:
# ---------------- 入口：选择待评审行，并发跑起来 ----------------
IDX_O = DF.index.tolist()
print(f"▶️ Oracle rows: {len(IDX_O)}")

# 你已有的 seed 逻辑（从 OUT_ORC 读旧结果回填 DF）保持不变
seed = _seed_from(OUT_ORC, ["oracle::answer_relevancy", "oracle::faithfulness"])
if seed is not None and len(seed) == len(DF):
    for c in seed.columns:
        DF[c] = seed[c].values

pending_o = [i for i in IDX_O if any(pd.isna(DF.loc[i, c]) for c in
                                    ["oracle::answer_relevancy", "oracle::faithfulness"])]
print(f"🔄 Processing {len(pending_o)} pending oracle rows...")


In [ ]:
# 异步启动
if pending_o:
    try:
        asyncio.run(_evaluate_rows_async(pending_o, DF))
    except RuntimeError:
        # 若在交互式环境已有事件循环（如 Jupyter），换一种启动方式
        loop = asyncio.get_event_loop()
        loop.run_until_complete(_evaluate_rows_async(pending_o, DF))
else:
    print("👍 Nothing to do; all oracle metrics already present.")


In [ ]:
import os, tempfile, shutil
import pandas as pd
import numpy as np

# 若你脚本里还没定义过原子落盘工具，保留这段；已定义则可删掉这段重复定义
def _atomic_write_parquet(df: pd.DataFrame, path: str):
    tmpdir = tempfile.mkdtemp(prefix="final_tmp_")
    tmppath = os.path.join(tmpdir, "part.parquet")
    try:
        df.to_parquet(tmppath, index=False)
        shutil.move(tmppath, path)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)

def save_all_metrics_one_file(
    df: pd.DataFrame,
    out_path: str,
    *,
    note: str | None = None,
) -> None:
    """
    将所有评估指标（Oracle + E5 + AIS）一次性保存到一个 Parquet 文件，并打印简要概况。
    只写一个文件（out_path），不额外生成侧车文件。
    """
    metrics_oracle = ["oracle::answer_relevancy", "oracle::faithfulness"]
    metrics_e5     = ["e5_align_q_to_a", "e5_align_q_oracle_to_a", "e5_align_q_retrieval_to_a"]
    metrics_ais    = ["ais_oracle", "ais_retrieval", "ais"]

    # 转数值、容错 NaN
    for c in metrics_oracle + metrics_e5 + metrics_ais:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # 原子落盘（只保存这一个文件）
    _atomic_write_parquet(df, out_path)
    print(f"💾 Saved ALL metrics -> {out_path}")

    # —— 概况打印（可选但很快）——
    n_total = len(df)
    print(f"🧮 Rows total: {n_total}")

    def _report(cols, name):
        present = [c for c in cols if c in df.columns]
        if not present:
            print(f"⚠️ {name}: columns not found: {cols}")
            return
        mask = df[present].notna().all(axis=1)
        n_done = int(mask.sum())
        rate = (n_done / n_total) if n_total else 0.0
        print(f"📊 {name}: done {n_done}/{n_total} ({rate:.1%}) on {present}")
        if n_done > 0:
            desc = df.loc[mask, present].describe(percentiles=[0.1,0.25,0.5,0.75,0.9])
            print(desc.to_string(float_format=lambda x: f"{x:0.4f}"))

    _report(metrics_oracle, "Oracle metrics")
    _report(metrics_e5,     "E5 alignments")
    _report(metrics_ais,    "AIS scores")

    if note:
        print("📝", note)

In [ ]:
# -------- 优化的E5对齐计算 --------
print("🔄 Computing E5 alignments...")

def _cos_sim_fast(a, b):
    """快速余弦相似度计算"""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

def _mk_query(text, ctx=None):
    if ctx and isinstance(ctx, (list, tuple)) and len(ctx) > 0:
        joined = " ".join([c for c in ctx if isinstance(c, str) and c.strip()][:3])  # 限制上下文长度
        return f"query: {text} [SEP] {joined}"
    return f"query: {text}"

def _mk_passage(ans):
    return f"passage: {ans}"

# 批量编码优化
q_texts = [_mk_query(str(q)) for q in DF["question"]]
a_texts = [_mk_passage(str(a)) for a in DF["answer"]]

print("🔄 Encoding questions and answers...")
q_e5 = e5.encode(q_texts, normalize_embeddings=True, batch_size=EMBEDDING_BATCH_SIZE,
                 show_progress_bar=True, convert_to_numpy=True)
a_e5 = e5.encode(a_texts, normalize_embeddings=True, batch_size=EMBEDDING_BATCH_SIZE,
                 show_progress_bar=True, convert_to_numpy=True)

# 向量化余弦相似度计算
DF["e5_align_q_to_a"] = np.sum(q_e5 * a_e5, axis=1)  # 已归一化，点积即余弦相似度

print("🔄 Encoding with oracle context...")
qo_texts = [_mk_query(str(q), ctx) for q, ctx in zip(DF["question"], DF["_ref_topk"])]
qo_e5 = e5.encode(qo_texts, normalize_embeddings=True, batch_size=EMBEDDING_BATCH_SIZE,
                  show_progress_bar=True, convert_to_numpy=True)
DF["e5_align_q_oracle_to_a"] = np.sum(qo_e5 * a_e5, axis=1)

print("🔄 Encoding with retrieval context...")
qr_texts = [_mk_query(str(q), ctx) for q, ctx in zip(DF["question"], DF["_ctx_topk"])]
qr_e5 = e5.encode(qr_texts, normalize_embeddings=True, batch_size=EMBEDDING_BATCH_SIZE,
                  show_progress_bar=True, convert_to_numpy=True)
DF["e5_align_q_retrieval_to_a"] = np.sum(qr_e5 * a_e5, axis=1)

print("✔️ E5 alignment done.")


In [ ]:
# -------- 优化的AIS计算 --------
def sent_split_fast(text: str, min_len: int = 12):
    """优化的句子分割"""
    if not isinstance(text, str) or not text.strip():
        return []
    # 使用更简单但更快的分割方式
    parts = re.split(r'[.!?。！？]\s+', text.strip())
    return [p.strip() for p in parts if len(p.strip()) >= min_len]

def ais_e5_batch(answers, sources_list, sent_th: float = 0.70):
    """批量AIS计算"""
    results = []
    all_sents = []
    all_sources = []
    sent_to_answer = []

    # 收集所有句子和源文档
    for i, (answer, sources) in enumerate(zip(answers, sources_list)):
        sents = sent_split_fast(answer or "")
        if not sents:
            results.append(0.0)
            continue

        srcs = _ensure_list_str(sources)
        if not srcs:
            results.append(0.0)
            continue

        all_sents.extend(sents)
        all_sources.extend(srcs)
        sent_to_answer.extend([i] * len(sents))

    if not all_sents:
        return results

    # 批量编码
    print("🔄 Encoding sentences for AIS...")
    sent_emb = e5.encode(all_sents, normalize_embeddings=True,
                        batch_size=EMBEDDING_BATCH_SIZE, show_progress_bar=True)
    src_emb = e5.encode(list(set(all_sources)), normalize_embeddings=True,
                       batch_size=EMBEDDING_BATCH_SIZE, show_progress_bar=True)

    # 计算相似度矩阵
    sims = np.dot(sent_emb, src_emb.T)
    max_sims = (np.max(sims, axis=1) + 1.0) / 2.0  # 归一化到[0,1]

    # 按答案聚合结果
    answer_results = {}
    for sent_idx, answer_idx in enumerate(sent_to_answer):
        if answer_idx not in answer_results:
            answer_results[answer_idx] = []
        answer_results[answer_idx].append(max_sims[sent_idx] >= sent_th)

    # 填充所有结果
    final_results = []
    result_idx = 0
    for i in range(len(answers)):
        if i in answer_results:
            final_results.append(float(np.mean(answer_results[i])))
        else:
            final_results.append(results[result_idx])
            result_idx += 1

    return final_results

print("🔄 Computing AIS scores...")

In [ ]:
# 批量计算AIS
oracle_ais = ais_e5_batch(DF["answer"].tolist(), DF["_ref_topk"].tolist())
DF["ais_oracle"] = oracle_ais

retrieval_ais = ais_e5_batch(DF["answer"].tolist(), DF["_ctx_topk"].tolist())
DF["ais_retrieval"] = retrieval_ais

# 合并AIS
DF["ais"] = np.where(pd.isna(DF["ais_retrieval"]), DF["ais_oracle"], DF["ais_retrieval"])

print("✔️ AIS done.")

In [ ]:
OUT_ALL = os.path.splitext(OUT_ORC)[0] + "_ALL_metrics.parquet"  # 或自定义路径
save_all_metrics_one_file(DF, OUT_ALL, note="final save")

In [ ]:
import os, re, tempfile, shutil
import pandas as pd
import numpy as np

# ======= 1) 填你的两个路径（照你给的保持不变）=======
PATH_RETR = "/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_scored.retrieval.1.1.parquet"
PATH_ALL  = "/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_scored.oracle.1.1_ALL_metrics.parquet"

# 自动生成合并输出路径（同目录）
def _derive_merged_path(path_all: str) -> str:
    base, ext = os.path.splitext(path_all)
    return base.replace(".oracle.", ".merged.") + ext if ".oracle." in base else base + "_merged" + ext

PATH_MERGED = _derive_merged_path(PATH_ALL)

In [ ]:
# ======= 2) 小工具 =======
def atomic_write_parquet(df: pd.DataFrame, path: str):
    tmpdir = tempfile.mkdtemp(prefix="merge_tmp_")
    tmppath = os.path.join(tmpdir, "part.parquet")
    try:
        df.to_parquet(tmppath, index=False)
        shutil.move(tmppath, path)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)

def inspect_columns(df: pd.DataFrame, name: str):
    print(f"\n== {name} ==")
    print(f"rows={len(df)}, cols={len(df.columns)}")
    print("columns:", list(df.columns))

def choose_join_key(df_left: pd.DataFrame, df_right: pd.DataFrame):
    # 优先策略
    candidates = [
        ["qid","model"],
        ["qid","doc_id","model"],
        ["qid"],
        ["doc_id","question"],
        ["doc_id"],
    ]
    for key in candidates:
        if all(c in df_left.columns for c in key) and all(c in df_right.columns for c in key):
            # 要求两边 key 唯一（避免一对多）
            if not df_left.duplicated(subset=key).any() and not df_right.duplicated(subset=key).any():
                return key
    # 自动猜测：找两边共同列中，能唯一标识两边的最小键
    common = [c for c in df_left.columns if c in df_right.columns]
    # 粗暴地试一下最多三列的组合
    for k in common:
        if not df_left.duplicated(subset=[k]).any() and not df_right.duplicated(subset=[k]).any():
            return [k]
    # 兜底：不强制唯一，退化到 ['qid'] 或共同列首个
    if "qid" in common:
        return ["qid"]
    if common:
        return [common[0]]
    raise ValueError("无法找到可用的合并键（没有共同列）")

In [ ]:
# ======= 3) 读取并展示列差异 =======
DF_RETR = pd.read_parquet(PATH_RETR)
DF_ALL  = pd.read_parquet(PATH_ALL)

inspect_columns(DF_RETR, "RAGAS(retrieval) file")
inspect_columns(DF_ALL,  "ALL-metrics (oracle+E5+AIS) file")

set_retr = set(DF_RETR.columns)
set_all  = set(DF_ALL.columns)
only_in_retr = sorted(set_retr - set_all)
only_in_all  = sorted(set_all  - set_retr)
common_cols  = sorted(set_retr & set_all)

print("\n— 列差异 —")
print("only_in_retr (将新增这些列):", only_in_retr[:50], "..." if len(only_in_retr)>50 else "")
print("only_in_all :", only_in_all[:50],  "..." if len(only_in_all)>50 else "")
print("common_cols :", common_cols[:50],  "..." if len(common_cols)>50 else "")

In [ ]:
import json, math
import numpy as np
import pandas as pd
import shutil, os, tempfile

# —— 规范化任意 Python 值为可比较的 JSON ——
def _canon_value(x):
    # 处理 None/NaN/Inf
    if x is None:
        return {"__type__": "none"}
    if isinstance(x, float):
        if math.isnan(x):
            return {"__type__": "nan"}
        if math.isinf(x):
            return {"__type__": "inf", "sign": 1 if x > 0 else -1}
    # ndarray -> list
    if isinstance(x, np.ndarray):
        x = x.tolist()
    # 递归处理 list/tuple/dict
    if isinstance(x, (list, tuple)):
        return {"__type__": "list", "data": [_canon_value(v) for v in x]}
    if isinstance(x, dict):
        # 按 key 排序保证稳定
        return {"__type__": "dict", "data": sorted([(k, _canon_value(v)) for k, v in x.items()], key=lambda kv: kv[0])}
    # 其他原子类型
    return {"__type__": "atom", "data": x}

def _norm_series_as_json_strings(s: pd.Series) -> np.ndarray:
    # 将整列转为 JSON 字符串（稳定可比较）
    return np.array(
        [json.dumps(_canon_value(v), ensure_ascii=False, sort_keys=True) for v in s.tolist()],
        dtype=object
    )

def _col_mismatch_count(a: pd.Series, b: pd.Series, rtol=1e-6, atol=1e-8) -> int:
    # 数值列：用 isclose（含 NaN 对齐）
    if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
        a_vals = a.astype(float).to_numpy(copy=False)
        b_vals = b.astype(float).to_numpy(copy=False)
        a_nan = np.isnan(a_vals)
        b_nan = np.isnan(b_vals)
        # NaN 和 NaN 视为相等
        both_nan = a_nan & b_nan
        cmp_mask = ~(both_nan)
        if not cmp_mask.any():
            return 0
        ok = np.isclose(a_vals[cmp_mask], b_vals[cmp_mask], rtol=rtol, atol=atol, equal_nan=True)
        return int((~ok).sum())
    # 其他类型：规范化后逐元素字符串比较
    na = _norm_series_as_json_strings(a)
    nb = _norm_series_as_json_strings(b)
    return int(np.sum(na != nb))

# ========== 读取并对齐 ==========
PATH_RETR = "/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_scored.retrieval.1.1.parquet"
PATH_ALL  = "/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_scored.oracle.1.1_ALL_metrics.parquet"

df_retr = pd.read_parquet(PATH_RETR)
df_all  = pd.read_parquet(PATH_ALL)

# 列顺序对齐（以 ALL 为准）
df_retr = df_retr[df_all.columns]

# ========== 逐列比较 ==========
mismatch = {}
for c in df_all.columns:
    mismatch[c] = _col_mismatch_count(df_retr[c], df_all[c])

diff_cols = [c for c, k in mismatch.items() if k > 0]
print(f"🔎 不一致的列数量: {len(diff_cols)}")
if diff_cols:
    print("差异列（最多 20 个）:", diff_cols[:20])
else:
    print("✅ 两个文件在全部 24 列上内容完全一致。")

# ========== 若一致：直接复制为 merged；否则做“检索列优先”合并 ==========
def _atomic_write_parquet(df: pd.DataFrame, path: str):
    tmpdir = tempfile.mkdtemp(prefix="merge_out_")
    tmppath = os.path.join(tmpdir, "part.parquet")
    try:
        df.to_parquet(tmppath, index=False)
        shutil.move(tmppath, path)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)

def _derive_merged_path(path_all: str) -> str:
    base, ext = os.path.splitext(path_all)
    if ".oracle." in base:
        base = base.replace(".oracle.", ".merged.")
    else:
        base = base + ".merged"
    return base + ext

PATH_MERGED = _derive_merged_path(PATH_ALL)

if not diff_cols:
    shutil.copy2(PATH_ALL, PATH_MERGED)
    print(f"💾 已复制 ALL 为合并结果 -> {PATH_MERGED}")
else:
    # 仅覆盖 retrieval::*（需要的话可扩展到 retrieved_ctx / citations）
    retr_cols = [c for c in df_all.columns if c.startswith("retrieval::")]
    merged = df_all.copy()
    for c in retr_cols:
        merged[c] = df_retr[c].combine_first(merged[c])
    _atomic_write_parquet(merged, PATH_MERGED)
    print(f"💾 已按检索列优先生成合并文件 -> {PATH_MERGED}")
    # 打印每个差异列的不同元素计数（前 20 个）
    print("📌 差异计数（前 20 个）：", {k: mismatch[k] for k in diff_cols[:20]})


In [ ]:
import json, math, numpy as np, pandas as pd

# --- 通用规范化：把任意值（含 list/ndarray/dict/NaN）转为可比较的 JSON 字符串 ---
def _canon_value(x):
    if x is None: return {"__t__":"none"}
    if isinstance(x, float):
        if math.isnan(x): return {"__t__":"nan"}
        if math.isinf(x): return {"__t__":"inf","s":1 if x>0 else -1}
    if isinstance(x, np.ndarray): x = x.tolist()
    if isinstance(x, (list, tuple)): return {"__t__":"list","d":[_canon_value(v) for v in x]}
    if isinstance(x, dict): return {"__t__":"dict","d":sorted([(k,_canon_value(v)) for k,v in x.items()], key=lambda kv: kv[0])}
    return {"__t__":"atom","d":x}

def _series_to_canon_str(s: pd.Series) -> np.ndarray:
    return np.array([json.dumps(_canon_value(v), ensure_ascii=False, sort_keys=True) for v in s.tolist()], dtype=object)

def _series_equal(a: pd.Series, b: pd.Series, rtol=1e-6, atol=1e-8) -> bool:
    # 两边都数值 => allclose（含 NaN 相等）
    if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
        av = a.to_numpy(dtype=float, copy=False)
        bv = b.to_numpy(dtype=float, copy=False)
        return np.allclose(av, bv, rtol=rtol, atol=atol, equal_nan=True)
    # 否则做 JSON 规范化再逐元素对比
    av = _series_to_canon_str(a)
    bv = _series_to_canon_str(b)
    return np.array_equal(av, bv)

def _series_mismatch_count(a: pd.Series, b: pd.Series, rtol=1e-6, atol=1e-8) -> int:
    if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
        av = a.to_numpy(dtype=float, copy=False); bv = b.to_numpy(dtype=float, copy=False)
        ok = np.isclose(av, bv, rtol=rtol, atol=atol, equal_nan=True)
        return int((~ok).sum())
    av = _series_to_canon_str(a); bv = _series_to_canon_str(b)
    return int(np.sum(av != bv))

# ===== 重新做校验（替换你原来的“① 严格校验合并是否正确”那块） =====
P_RETR   = "/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_scored.retrieval.1.1.parquet"
P_ALL    = "/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_scored.oracle.1.1_ALL_metrics.parquet"
P_MERGED = "/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_scored.merged.1.1_ALL_metrics.parquet"

df_r = pd.read_parquet(P_RETR)
df_a = pd.read_parquet(P_ALL)
df_m = pd.read_parquet(P_MERGED)

retr_cols = [c for c in df_m.columns if c.startswith("retrieval::")]
base_cols = [c for c in df_m.columns if c not in retr_cols]

# 1) merged 的检索列 == retrieval 文件
for c in retr_cols:
    assert _series_equal(df_m[c], df_r[c]), f"Merge check failed for {c}"

# 2) merged 的非检索列 == ALL_metrics 文件
bad = []
for c in base_cols:
    if not _series_equal(df_m[c], df_a[c]):
        bad.append((c, _series_mismatch_count(df_m[c], df_a[c])))
if bad:
    print("⚠️ 下列非检索列在合并后与 ALL 不一致（列名, 不同元素数）:", bad[:20])
    raise AssertionError("Base columns differ after merge.")
print("✅ Merge integrity checks passed.")


In [ ]:
OUT_ALL = "/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_scored.merged.1.1_ALL_metrics.parquet"

In [ ]:
import pandas as pd, numpy as np

df = pd.read_parquet(OUT_ALL)

def quick_report(df):
    blocks = {
        "Oracle": ["oracle::answer_relevancy","oracle::faithfulness"],
        "Retrieval": ["retrieval::answer_relevancy","retrieval::faithfulness",
                      "retrieval::context_precision","retrieval::context_recall"],
        "E5": ["e5_align_q_to_a","e5_align_q_oracle_to_a","e5_align_q_retrieval_to_a"],
        "AIS": ["ais_oracle","ais_retrieval","ais"],
    }
    print(f"Rows: {len(df)}")
    for name, cols in blocks.items():
        present = [c for c in cols if c in df.columns]
        if not present:
            continue
        mask = df[present].notna().all(axis=1)
        print(f"\n📊 {name} — done {int(mask.sum())}/{len(df)}")
        if mask.any():
            print(df.loc[mask, present].describe().to_string(float_format=lambda x: f"{x:0.4f}"))

quick_report(df)

In [ ]:
metrics = ["retrieval::answer_relevancy","retrieval::faithfulness",
           "retrieval::context_precision","retrieval::context_recall"]
present = [m for m in metrics if m in df.columns]
by_model = (df.groupby("model")[present]
              .agg(["mean","std","count"])
              .sort_values((present[0],"mean"), ascending=False))
by_model.to_csv(OUT_ALL.replace(".parquet","_retrieval_by_model.csv"))
print("Saved:", OUT_ALL.replace(".parquet","_retrieval_by_model.csv"))
by_model.head()


In [ ]:
long = (df.melt(id_vars=[c for c in ["qid","model","paper_id","question_id"] if c in df.columns],
                value_vars=present,
                var_name="metric", value_name="score")
          .dropna(subset=["score"]))
long.to_parquet(OUT_ALL.replace(".parquet","_retrieval_long.parquet"), index=False)
print("Saved:", OUT_ALL.replace(".parquet","_retrieval_long.parquet"))
long.head()


In [ ]:
import pandas as pd
import numpy as np

# ❶ 路径：用你合并后的文件
OUT_ALL = "/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_scored.merged.1.1_ALL_metrics.parquet"

df = pd.read_parquet(OUT_ALL)

# ❷ 指标分组
blocks = {
    "Oracle": ["oracle::answer_relevancy","oracle::faithfulness"],
    "Retrieval": ["retrieval::answer_relevancy","retrieval::faithfulness",
                  "retrieval::context_precision","retrieval::context_recall"],
    "E5": ["e5_align_q_to_a","e5_align_q_oracle_to_a","e5_align_q_retrieval_to_a"],
    "AIS": ["ais_oracle","ais_retrieval","ais"],
}

# ❸ 模型显示顺序（若你的列正好是 'M0'...'M3'）
model_order = ["M0","M1","M2","M3"]
if "model" in df.columns:
    df["model"] = pd.Categorical(df["model"], categories=model_order, ordered=True)

def summarize_by_model(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """
    对每个 metric 单独 dropna，再按 model 计算 n/mean/std，最后拼成宽表。
    """
    out = None
    for c in cols:
        # 单列的 per-model 统计
        s = df[["model", c]].dropna(subset=[c]).groupby("model")[c].agg(n="count", mean="mean", std="std")
        # 列层级命名
        s.columns = pd.MultiIndex.from_product([[c], s.columns])
        out = s if out is None else out.join(s, how="outer")
    # 排序并美化
    out = out.sort_index()
    # 可选：四舍五入
    for c in cols:
        for stat in ["mean","std"]:
            if (c, stat) in out.columns:
                out[(c, stat)] = out[(c, stat)].astype(float).round(4)
    return out

# ❹ 逐块汇总 + 导出
for name, cols in blocks.items():
    present = [c for c in cols if c in df.columns]
    if not present:
        print(f"⚠️ {name}: 指标列不存在，跳过。")
        continue
    tbl = summarize_by_model(df, present)
    print(f"\n==== {name} (by model) ====")
    print(tbl.to_string())
    # 导出 CSV
    out_csv = OUT_ALL.replace(".parquet", f"_{name.lower()}_by_model.csv")
    # 为了 CSV 平铺好看，展平列名
    flat = tbl.copy()
    flat.columns = [f"{m}__{stat}" for m, stat in flat.columns]
    flat.to_csv(out_csv)
    print("Saved ->", out_csv)


In [ ]:
import pandas as pd
import numpy as np

# 合并后的权威文件
OUT_ALL = "/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_scored.merged.1.1_ALL_metrics.parquet"
df = pd.read_parquet(OUT_ALL)

# 各指标分组
blocks = {
    "Oracle": ["oracle::answer_relevancy","oracle::faithfulness"],
    "Retrieval": ["retrieval::answer_relevancy","retrieval::faithfulness",
                  "retrieval::context_precision","retrieval::context_recall"],
    "E5": ["e5_align_q_to_a","e5_align_q_oracle_to_a","e5_align_q_retrieval_to_a"],
    "AIS": ["ais_oracle","ais_retrieval","ais"],
}

# 模型顺序
order = ["M0","M1","M2","M3"]
if "model" in df.columns:
    df["model"] = pd.Categorical(df["model"], categories=order, ordered=True)

for name, cols in blocks.items():
    present = [c for c in cols if c in df.columns]
    if not present:
        continue
    # 按模型计算中位数
    med = df.groupby("model", observed=True)[present].median(numeric_only=True)
    med = med.reindex(order)  # 固定顺序
    print(f"\n=== {name} (median) ===")
    print(med.round(4).to_string())


In [ ]:
import pandas as pd

# 合并后的权威文件
OUT_ALL = "/Volumes/main/default/thesis_project/Evaluation/Govern_Eval/govern_scored.merged.1.1_ALL_metrics.parquet"
df = pd.read_parquet(OUT_ALL)

# 指标分组
blocks = {
    "Oracle": ["oracle::answer_relevancy","oracle::faithfulness"],
    "Retrieval": ["retrieval::answer_relevancy","retrieval::faithfulness",
                  "retrieval::context_precision","retrieval::context_recall"],
    "E5": ["e5_align_q_to_a","e5_align_q_oracle_to_a","e5_align_q_retrieval_to_a"],
    "AIS": ["ais_oracle","ais_retrieval","ais"],
}

# 模型顺序
order = ["M0","M1","M2","M3"]
if "model" in df.columns:
    df["model"] = pd.Categorical(df["model"], categories=order, ordered=True)

for name, cols in blocks.items():
    present = [c for c in cols if c in df.columns]
    if not present:
        continue
    mean_tbl = (df.groupby("model", observed=True)[present]
                  .mean(numeric_only=True)
                  .reindex(order))
    print(f"\n=== {name} (mean) ===")
    print(mean_tbl.round(4).to_string())


In [ ]:
# QASPER 数据打印
import pandas as pd

# 合并后的权威文件
OUT_ALL = "/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.scored.full.1.16.parquet"
df = pd.read_parquet(OUT_ALL)

# 指标分组
blocks = {
    "Oracle": ["oracle::answer_relevancy","oracle::faithfulness"],
    "Retrieval": ["retrieval::answer_relevancy","retrieval::faithfulness",
                  "retrieval::context_precision","retrieval::context_recall"],
    "E5": ["e5_align_q_to_a","e5_align_q_oracle_to_a","e5_align_q_retrieval_to_a"],
    "AIS": ["ais_oracle","ais_retrieval","ais"],
}

# 模型顺序
order = ["M0","M1","M2","M3"]
if "model" in df.columns:
    df["model"] = pd.Categorical(df["model"], categories=order, ordered=True)

for name, cols in blocks.items():
    present = [c for c in cols if c in df.columns]
    if not present:
        continue
    mean_tbl = (df.groupby("model", observed=True)[present]
                  .mean(numeric_only=True)
                  .reindex(order))
    print(f"\n=== {name} (mean) ===")
    print(mean_tbl.round(4).to_string())